# Fine-tune `typhoon-ai/typhoon2.5-qwen3-4b` for Thai RAG QA

Standalone notebook สำหรับเทรนบน server โดยให้แก้เฉพาะ path และ training knobs ใน cell ต้น ๆ ได้ทันที

In [ ]:
from pathlib import Path

# Editable path configuration.
# แก้ cell นี้เมื่อย้าย notebook ไปใช้บน server หรือเปลี่ยนตำแหน่ง model / data.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_JSON_PATH = PROJECT_ROOT / "data" / "train" / "train_set.json"

# รองรับทั้ง Hugging Face model id และ local path บน server
MODEL_NAME_OR_PATH = "typhoon-ai/typhoon2.5-qwen3-4b"
# MODEL_NAME_OR_PATH = Path("/mnt/models/typhoon2.5-qwen3-4b")

# รองรับทั้ง local path และ model id สำหรับ retrieval / semantic score
EMBED_MODEL_NAME_OR_PATH = PROJECT_ROOT / "weight" / "bge-m3"
# EMBED_MODEL_NAME_OR_PATH = "BAAI/bge-m3"

OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "typhoon25_qwen3_4b_rag_qa_qlora"
CACHE_DIR = None
# CACHE_DIR = Path("/mnt/cache/huggingface")

TRAIN_JSON_PATH, MODEL_NAME_OR_PATH, EMBED_MODEL_NAME_OR_PATH, OUTPUT_DIR, CACHE_DIR

In [ ]:
# Editable training / evaluation configuration.
MAX_SEQ_LEN = 4096
MAX_NEW_TOKENS = 384
RETRIEVAL_TOP_K = 10
REFERENCE_TOP_N = 3
VAL_DOC_RATIO = 0.2
SEED = 42

TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 3
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 2

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

# Optional smoke-test caps. Set to integers for quick server checks.
DEBUG_MAX_TRAIN_SAMPLES = None
DEBUG_MAX_VAL_SAMPLES = None

MAX_SEQ_LEN, MAX_NEW_TOKENS, RETRIEVAL_TOP_K, REFERENCE_TOP_N, VAL_DOC_RATIO, SEED

In [ ]:
import importlib.util
import subprocess
import sys

PACKAGE_SPECS = {
    "transformers": "transformers>=4.51.0",
    "datasets": "datasets>=2.19.0",
    "accelerate": "accelerate>=0.34.0",
    "peft": "peft>=0.13.0",
    "bitsandbytes": "bitsandbytes>=0.43.1",
    "sentence_transformers": "sentence-transformers>=3.0.1",
    "pythainlp": "pythainlp>=5.0.4",
    "rouge_score": "rouge-score>=0.1.2",
    "pandas": "pandas>=2.0.0",
    "numpy": "numpy>=1.24.0",
}

missing = [spec for module_name, spec in PACKAGE_SPECS.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])
else:
    print("All required packages are already installed.")

In [ ]:
import json
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from pythainlp.tokenize import word_tokenize
from rouge_score import rouge_scorer
from rouge_score.tokenizers import Tokenizer
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

CACHE_DIR = Path(CACHE_DIR).expanduser().resolve() if CACHE_DIR is not None else None
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()
TRAIN_JSON_PATH = Path(TRAIN_JSON_PATH).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if CACHE_DIR is not None:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(CACHE_DIR)
    os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR)
    os.environ["HF_DATASETS_CACHE"] = str(CACHE_DIR / "datasets")

FINAL_ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
VAL_PREDICTIONS_PATH = OUTPUT_DIR / "val_predictions.csv"
VAL_METRICS_PATH = OUTPUT_DIR / "validation_metrics.json"

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_model_source(value) -> str:
    if isinstance(value, Path):
        return str(value.expanduser().resolve())
    raw = str(value)
    candidate = Path(raw).expanduser()
    is_probable_local_path = (
        raw.startswith(("~", ".", "/"))
        or re.match(r"^[A-Za-z]:[\\/]", raw) is not None
        or candidate.exists()
    )
    return str(candidate.resolve()) if is_probable_local_path else raw

def print_runtime_config() -> None:
    print(f"TRAIN_JSON_PATH={TRAIN_JSON_PATH}")
    print(f"MODEL_NAME_OR_PATH={resolve_model_source(MODEL_NAME_OR_PATH)}")
    print(f"EMBED_MODEL_NAME_OR_PATH={resolve_model_source(EMBED_MODEL_NAME_OR_PATH)}")
    print(f"OUTPUT_DIR={OUTPUT_DIR}")
    print(f"CACHE_DIR={CACHE_DIR}")
    print(f"CUDA available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU count={torch.cuda.device_count()}")
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"  GPU {idx}: {torch.cuda.get_device_name(idx)} | {props.total_memory / 1024**3:.2f} GB")

set_global_seed(SEED)
print_runtime_config()

SYSTEM_PROMPT = (
    "คุณเป็นผู้ช่วยตอบคำถามจากบันทึกการประชุมรัฐสภาไทย\n"
    "กฎ:\n"
    "- ตอบจากข้อมูลในเอกสารที่ให้มาเท่านั้น\n"
    "- ใช้ภาษาไทย\n"
    "- ไม่ต้องขึ้นต้นด้วยคำว่า \"ตอบ:\" หรือ \"คำตอบ:\"\n"
    "- หากข้อมูลไม่เพียงพอ ให้ตอบว่า \"ไม่พบข้อมูลในเอกสาร\""
)

def build_user_prompt(context: str, query: str) -> str:
    return f"เอกสาร:\n{context}\n\nคำถาม: {query}"

def cache_dir_as_str() -> str | None:
    return None if CACHE_DIR is None else str(CACHE_DIR)

In [ ]:
def load_training_data(train_json_path: Path):
    with train_json_path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    docs = data.get("docs") or []
    queries = data.get("queries") or []
    if not docs or not queries:
        raise ValueError("Training JSON must contain non-empty 'docs' and 'queries'.")
    doc_lookup = {doc["doc_id"]: doc for doc in docs}
    return docs, queries, doc_lookup

def grouped_doc_split(queries, val_ratio: float, seed: int):
    doc_ids = sorted({q["doc_id"] for q in queries})
    if len(doc_ids) < 2:
        raise ValueError("Need at least 2 unique doc_id values for grouped validation split.")
    rng = random.Random(seed)
    rng.shuffle(doc_ids)
    val_doc_count = max(1, int(round(len(doc_ids) * val_ratio)))
    val_doc_count = min(val_doc_count, len(doc_ids) - 1)
    val_doc_ids = set(doc_ids[:val_doc_count])
    train_doc_ids = set(doc_ids[val_doc_count:])
    train_queries = [q for q in queries if q["doc_id"] in train_doc_ids]
    val_queries = [q for q in queries if q["doc_id"] in val_doc_ids]
    return train_queries, val_queries, train_doc_ids, val_doc_ids

def ordered_ref_paragraphs(doc: dict, refs: list[str]):
    ref_set = set(refs or [])
    ordered = [p for p in doc["paragraphs"] if p["para_id"] in ref_set]
    found = {p["para_id"] for p in ordered}
    missing = [ref for ref in refs if ref not in found]
    return ordered, missing

def build_raw_samples(queries, doc_lookup):
    samples = []
    missing_ref_records = []
    for q in queries:
        doc = doc_lookup[q["doc_id"]]
        ordered_refs, missing_refs = ordered_ref_paragraphs(doc, q.get("refs", []))
        context_lines = [
            f"[{paragraph['para_id']}] {paragraph['text'].strip()}"
            for paragraph in ordered_refs
            if paragraph.get("text", "").strip()
        ]
        if missing_refs:
            missing_ref_records.append({"ID": q["ID"], "missing_refs": missing_refs})
        samples.append(
            {
                "ID": q["ID"],
                "doc_id": q["doc_id"],
                "query": q["query"].strip(),
                "answer": q["abstractive"].strip(),
                "context": "\n".join(context_lines) if context_lines else "(ไม่มีข้อมูลอ้างอิง)",
                "gold_refs": q.get("refs", []),
            }
        )
    return samples, missing_ref_records

docs, queries, doc_lookup = load_training_data(TRAIN_JSON_PATH)
train_queries, val_queries, train_doc_ids, val_doc_ids = grouped_doc_split(queries, VAL_DOC_RATIO, SEED)

if DEBUG_MAX_TRAIN_SAMPLES is not None:
    train_queries = train_queries[:DEBUG_MAX_TRAIN_SAMPLES]
if DEBUG_MAX_VAL_SAMPLES is not None:
    val_queries = val_queries[:DEBUG_MAX_VAL_SAMPLES]

train_raw_samples, train_missing_refs = build_raw_samples(train_queries, doc_lookup)
val_raw_samples, val_missing_refs = build_raw_samples(val_queries, doc_lookup)
val_doc_lookup = {doc_id: doc_lookup[doc_id] for doc_id in val_doc_ids}

print(f"Loaded {len(docs)} docs and {len(queries)} queries from {TRAIN_JSON_PATH}")
print(f"Train docs={len(train_doc_ids)} | Val docs={len(val_doc_ids)}")
print(f"Train samples={len(train_raw_samples)} | Val samples={len(val_raw_samples)}")
print(f"Missing train refs={len(train_missing_refs)} | Missing val refs={len(val_missing_refs)}")

for sample in train_raw_samples[:2]:
    print("=" * 100)
    print(f"ID={sample['ID']} | doc_id={sample['doc_id']}")
    print(f"QUERY: {sample['query']}")
    print("CONTEXT PREVIEW:")
    print(sample['context'][:1500])
    print("ANSWER PREVIEW:")
    print(sample['answer'][:600])

In [ ]:
MODEL_SOURCE = resolve_model_source(MODEL_NAME_OR_PATH)
TOKENIZER = AutoTokenizer.from_pretrained(
    MODEL_SOURCE,
    trust_remote_code=True,
    cache_dir=cache_dir_as_str(),
)
if TOKENIZER.pad_token is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.padding_side = "right"

def tokenize_supervised_sample(sample: dict, tokenizer, max_seq_len: int):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(sample['context'], sample['query'])},
    ]
    full_messages = prompt_messages + [{"role": "assistant", "content": sample['answer']}]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

    if len(full_ids) > max_seq_len:
        return None, {"ID": sample['ID'], "reason": f"overlength:{len(full_ids)}"}
    if full_ids[: len(prompt_ids)] != prompt_ids:
        return None, {"ID": sample['ID'], "reason": "prompt_alignment_failed"}

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        "ID": sample['ID'],
        "doc_id": sample['doc_id'],
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }, None

def build_tokenized_dataset(raw_samples, tokenizer, max_seq_len: int):
    encoded_rows = []
    dropped_rows = []
    for sample in raw_samples:
        encoded, dropped = tokenize_supervised_sample(sample, tokenizer, max_seq_len)
        if encoded is not None:
            encoded_rows.append(encoded)
        if dropped is not None:
            dropped_rows.append(dropped)
    if not encoded_rows:
        raise ValueError("No usable training samples remain after tokenization/truncation checks.")
    return Dataset.from_list(encoded_rows), dropped_rows

train_dataset, dropped_train = build_tokenized_dataset(train_raw_samples, TOKENIZER, MAX_SEQ_LEN)
val_dataset, dropped_val = build_tokenized_dataset(val_raw_samples, TOKENIZER, MAX_SEQ_LEN)

print(f"Tokenized train rows={len(train_dataset)} | dropped={len(dropped_train)}")
print(f"Tokenized val rows={len(val_dataset)} | dropped={len(dropped_val)}")
if dropped_train:
    print("Dropped train example preview:", dropped_train[:5])
if dropped_val:
    print("Dropped val example preview:", dropped_val[:5])

train_lengths = [len(row['input_ids']) for row in train_dataset]
val_lengths = [len(row['input_ids']) for row in val_dataset]
print(f"Train token length stats: min={min(train_lengths)} | max={max(train_lengths)} | avg={np.mean(train_lengths):.1f}")
print(f"Val token length stats: min={min(val_lengths)} | max={max(val_lengths)} | avg={np.mean(val_lengths):.1f}")

for sample in train_raw_samples[:2]:
    print("=" * 100)
    print(build_user_prompt(sample['context'], sample['query'])[:1800])
    print("ASSISTANT TARGET:")
    print(sample['answer'][:500])

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("QLoRA training requires a CUDA-enabled server runtime. Run this notebook on the target GPU server.")

class SupervisedDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        base_features = [
            {"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]}
            for f in features
        ]
        batch = self.tokenizer.pad(base_features, padding=True, return_tensors="pt")
        max_len = batch["input_ids"].shape[1]
        labels = []
        for feature in features:
            padded = feature["labels"] + ([-100] * (max_len - len(feature["labels"])))
            labels.append(padded)
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_SOURCE,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    cache_dir=cache_dir_as_str(),
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

bf16 = torch.cuda.is_bf16_supported()
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    bf16=bf16,
    fp16=not bf16,
    report_to="none",
    remove_unused_columns=False,
    optim="paged_adamw_8bit",
    seed=SEED,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=SupervisedDataCollator(TOKENIZER),
)

train_result = trainer.train()
trainer.save_state()
FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
TOKENIZER.save_pretrained(FINAL_ADAPTER_DIR)

print(train_result)
print(f"Saved adapter to {FINAL_ADAPTER_DIR}")

In [ ]:
class ThaiSpaceTokenizer(Tokenizer):
    def tokenize(self, text):
        return text.split(" ")

def tokenize_thai(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    return " ".join(word_tokenize(text, engine="newmm", keep_whitespace=False))

def parse_refs(value):
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    text = str(value).strip()
    return [item.strip() for item in text.split(",") if item.strip()] if text else []

def calculate_iou(pred_refs, gold_refs) -> float:
    pred_set = set(parse_refs(pred_refs))
    gold_set = set(parse_refs(gold_refs))
    if not gold_set:
        return 0.0
    return len(pred_set & gold_set) / len(pred_set | gold_set)

def calculate_final_score(metrics_dict) -> float:
    return (
        0.45 * metrics_dict["SS-score"]
        + 0.35 * metrics_dict["rougeL"]
        + 0.20 * metrics_dict["IoU"]
    )

def run_evaluation(sol_df: pd.DataFrame, pred_df: pd.DataFrame, semantic_model: SentenceTransformer):
    merged = sol_df.merge(pred_df, on="ID", suffixes=("_sol", "_pred"))
    merged["IoU"] = merged.apply(lambda row: calculate_iou(row["refs_pred"], row["refs_sol"]), axis=1)
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False, tokenizer=ThaiSpaceTokenizer())
    sol_tok = merged["abstractive_sol"].apply(tokenize_thai)
    pred_tok = merged["abstractive_pred"].apply(tokenize_thai)
    rouge_scores = [scorer.score(gold, pred)["rougeL"].fmeasure for gold, pred in zip(sol_tok, pred_tok)]
    merged["rougeL"] = rouge_scores

    texts = merged["abstractive_sol"].tolist() + merged["abstractive_pred"].tolist()
    embeddings = semantic_model.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    midpoint = len(texts) // 2
    ref_embeddings = embeddings[:midpoint]
    pred_embeddings = embeddings[midpoint:]
    semantic_scores = np.sum(ref_embeddings * pred_embeddings, axis=1)
    merged["SS-score"] = semantic_scores

    metrics = merged[["rougeL", "SS-score", "IoU"]].mean().to_dict()
    metrics["score"] = calculate_final_score(metrics)
    return metrics, merged

EMBED_SOURCE = resolve_model_source(EMBED_MODEL_NAME_OR_PATH)
embedder = SentenceTransformer(
    EMBED_SOURCE,
    device="cuda" if torch.cuda.is_available() else "cpu",
    cache_folder=cache_dir_as_str(),
)

doc_embedding_index = {}
for doc_id in tqdm(sorted(val_doc_lookup), desc="Embedding validation docs"):
    paragraphs = val_doc_lookup[doc_id]["paragraphs"]
    paragraph_texts = [paragraph["text"] for paragraph in paragraphs]
    embeddings = embedder.encode(
        paragraph_texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    doc_embedding_index[doc_id] = {
        "paragraphs": paragraphs,
        "embeddings": embeddings,
    }

def retrieve_paragraphs(doc_id: str, query: str, top_k: int):
    payload = doc_embedding_index[doc_id]
    query_embedding = embedder.encode(
        [query],
        batch_size=1,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    scores = payload["embeddings"] @ query_embedding
    top_indices = np.argsort(scores)[-top_k:][::-1].tolist()
    return [
        {
            "para_id": payload["paragraphs"][idx]["para_id"],
            "text": payload["paragraphs"][idx]["text"],
            "score": float(scores[idx]),
        }
        for idx in top_indices
    ]

def select_refs(retrieved, top_n: int):
    return [item["para_id"] for item in retrieved[:top_n]]

@torch.inference_mode()
def generate_answer(query: str, paragraphs: list[dict]):
    context = "\n".join(f"[{paragraph['para_id']}] {paragraph['text']}" for paragraph in paragraphs)
    if not context.strip():
        context = "(ไม่มีข้อมูลอ้างอิง)"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(context, query)},
    ]
    prompt_text = TOKENIZER.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = TOKENIZER(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(trainer.model.device)
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=TOKENIZER.pad_token_id,
        eos_token_id=TOKENIZER.eos_token_id,
    )
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer = TOKENIZER.decode(generated_ids, skip_special_tokens=True).strip()
    return answer or "ไม่พบข้อมูลในเอกสาร"

prediction_rows = []
gold_rows = []
for sample in tqdm(val_raw_samples, desc="Running validation inference"):
    retrieved = retrieve_paragraphs(sample["doc_id"], sample["query"], RETRIEVAL_TOP_K)
    predicted_refs = select_refs(retrieved, REFERENCE_TOP_N)
    predicted_answer = generate_answer(sample["query"], retrieved[:RETRIEVAL_TOP_K])
    prediction_rows.append(
        {
            "ID": sample["ID"],
            "abstractive": predicted_answer,
            "refs": ",".join(predicted_refs),
        }
    )
    gold_rows.append(
        {
            "ID": sample["ID"],
            "abstractive": sample["answer"],
            "refs": sample["gold_refs"],
        }
    )

pred_df = pd.DataFrame(prediction_rows)
gold_df = pd.DataFrame(gold_rows)
pred_df.to_csv(VAL_PREDICTIONS_PATH, index=False, encoding="utf-8")

metrics, merged_eval_df = run_evaluation(gold_df, pred_df, embedder)
with VAL_METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print(f"Saved validation predictions to {VAL_PREDICTIONS_PATH}")
print(f"Saved validation metrics to {VAL_METRICS_PATH}")
pred_df.head()